# Phase 3 - Autoencoder Training

This notebook trains or validates the Autoencoder used for dimensionality reduction. The model implementation stays in `src/autoencoder.py`; this notebook is only an interactive wrapper.

In [ ]:
# Cell 1 - Set up imports and make the project package importable from notebooks.
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG_PATH = PROJECT_ROOT / "configs" / "config.yaml"
print(f"Project root: {PROJECT_ROOT}")
print(f"Config path: {CONFIG_PATH}")


In [ ]:
# Cell 2 - Load the shared YAML config and resolve the main artifact paths.
from src.data_loading import load_config, resolve_project_path

config = load_config(CONFIG_PATH)
processed_dir = resolve_project_path(config["paths"]["processed_dir"], PROJECT_ROOT)
models_dir = resolve_project_path(config["paths"]["models_dir"], PROJECT_ROOT)
metrics_dir = resolve_project_path(config["paths"]["metrics_dir"], PROJECT_ROOT)
figures_dir = resolve_project_path(config["paths"]["figures_dir"], PROJECT_ROOT)

print(f"Expected feature count: {config['data']['expected_feature_count']}")
print(f"Expected classes: {config['data']['expected_num_classes']}")


In [ ]:
# Cell 3 - Confirm that Phase 2 processed arrays exist before training the Autoencoder.
processed_inputs = [
    processed_dir / "X_train.npy",
    processed_dir / "X_test.npy",
    processed_dir / "y_train.npy",
    processed_dir / "y_test.npy",
]

missing_inputs = [path for path in processed_inputs if not path.exists()]
if missing_inputs:
    raise FileNotFoundError(f"Missing Phase 2 artifacts: {missing_inputs}")

x_train = np.load(processed_dir / "X_train.npy", mmap_mode="r")
x_test = np.load(processed_dir / "X_test.npy", mmap_mode="r")
print(f"X_train shape: {x_train.shape}")
print(f"X_test shape: {x_test.shape}")


In [ ]:
# Cell 4 - Preview deterministic shuffled mini-batch composition for the first training epoch.
def make_validation_mask_local(n_rows, validation_split, seed):
    if not 0 < validation_split < 1:
        raise ValueError("validation_split must be between 0 and 1.")
    rng = np.random.default_rng(seed)
    validation_count = int(round(n_rows * validation_split))
    validation_indices = rng.choice(n_rows, size=validation_count, replace=False)
    validation_mask = np.zeros(n_rows, dtype=bool)
    validation_mask[validation_indices] = True
    return validation_mask

random_state = int(config["project"]["random_state"])
batch_size = int(config["autoencoder"]["batch_size"])
validation_split = float(config["autoencoder"]["validation_split"])
y_train = np.load(processed_dir / "y_train.npy", mmap_mode="r")
class_mapping = config["data"]["class_mapping"]
index_to_class = {index: label for label, index in class_mapping.items()}

validation_mask = make_validation_mask_local(x_train.shape[0], validation_split, random_state)
train_indices = np.flatnonzero(~validation_mask)
epoch_rng = np.random.default_rng(random_state + 1)
shuffled_train_indices = epoch_rng.permutation(train_indices)

batch_rows = []
for batch_id, start in enumerate(range(0, min(batch_size * 5, shuffled_train_indices.shape[0]), batch_size)):
    batch_indices = shuffled_train_indices[start : start + batch_size]
    labels, counts = np.unique(y_train[batch_indices], return_counts=True)
    composition = {
        index_to_class[int(label)]: int(count)
        for label, count in zip(labels, counts)
    }
    batch_rows.append({"batch_id": batch_id, "class_count": composition})

pd.DataFrame(batch_rows)


In [ ]:
# Cell 5 - Train the Autoencoder only if model artifacts are missing, or force retraining.
FORCE_RETRAIN = False
required_ae_outputs = [
    models_dir / "autoencoder.pt",
    models_dir / "encoder.pt",
    metrics_dir / "autoencoder_history.csv",
    metrics_dir / "autoencoder_reconstruction_error.json",
    figures_dir / "ae_loss_curve.png",
]

needs_training = FORCE_RETRAIN or not all(path.exists() for path in required_ae_outputs)
if needs_training:
    from src.autoencoder import train_autoencoder

    reconstruction_report = train_autoencoder(CONFIG_PATH, device_name="auto")
else:
    with open(metrics_dir / "autoencoder_reconstruction_error.json", "r", encoding="utf-8") as file:
        reconstruction_report = json.load(file)

print(json.dumps(reconstruction_report, indent=2))


In [ ]:
# Cell 6 - Review the training history to confirm that loss decreased and remained finite.
history = pd.read_csv(metrics_dir / "autoencoder_history.csv")
print(history.tail())
print(f"Train loss finite: {np.isfinite(history['train_loss']).all()}")
print(f"Validation loss finite: {np.isfinite(history['val_loss']).all()}")


In [ ]:
# Cell 7 - Load the saved checkpoint and verify that the encoder can produce 16-dimensional latent features.
try:
    import torch
    from src.autoencoder import Autoencoder
except ModuleNotFoundError as exc:
    print(f"Skipping checkpoint smoke test because this kernel is missing: {exc.name}")
    print("Select the 'Python (.venv multiclass IDS)' kernel to run this checkpoint validation.")
else:
    def load_torch_checkpoint(path):
        try:
            return torch.load(path, map_location="cpu", weights_only=False)
        except TypeError:
            return torch.load(path, map_location="cpu")

    checkpoint = load_torch_checkpoint(models_dir / "autoencoder.pt")
    metadata = checkpoint["metadata"]
    model = Autoencoder(input_dim=metadata["input_dim"], latent_dim=metadata["latent_dim"])
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    with torch.no_grad():
        sample = torch.as_tensor(np.array(x_test[:8], dtype=np.float32, copy=True))
        latent = model.encoder(sample)
        reconstructed = model(sample)

    print(f"Architecture metadata: {metadata}")
    print(f"Latent shape: {tuple(latent.shape)}")
    print(f"Reconstructed shape: {tuple(reconstructed.shape)}")


In [ ]:
# Cell 8 - Display the saved loss curve if the notebook frontend supports image display.
from IPython.display import Image, display

loss_curve_path = figures_dir / "ae_loss_curve.png"
if loss_curve_path.exists():
    display(Image(filename=str(loss_curve_path)))
else:
    print(f"Loss curve not found: {loss_curve_path}")
